In [0]:
dbutils.widgets.removeAll()

dbutils.widgets.dropdown("current_week", "auto", ["auto"] + [str(i) for i in range(1, 17)], "Current Week")
dbutils.widgets.dropdown("buybacks_used", "0", ["0", "1", "2"], "Buybacks Used")
dbutils.widgets.text(
    "used_teams_manual",
    "",
    "Teams Already Picked (comma-separated)"
)

In [0]:
used_teams_manual_raw = dbutils.widgets.get("used_teams_manual")

# Convert comma-separated text into a clean Python list
used_teams_manual = [
    t.strip() for t in used_teams_manual_raw.split(",") if t.strip() != ""
]

used_teams_manual

In [0]:
import pandas as pd

# Load original table (from your CSV import)
orig = spark.table("workspace.default.nfl_survivor_csv_export").toPandas()

# Clean column names to match predictions table
orig.columns = (
    orig.columns
    .str.replace(' ', '_', regex=False)
    .str.replace('[,;{}()\n\t=]', '_', regex=True)
)

# Filter rows that already had a Target value
orig_filled = orig[orig["Target"].notna()].copy()

# Load predicted table
pred = spark.table("workspace.default.nfl_survivor_predictions").toPandas()

# Clean predicted column names too
pred.columns = (
    pred.columns
    .str.replace(' ', '_', regex=False)
    .str.replace('[,;{}()\n\t=]', '_', regex=True)
)

# Replace Target with Predicted_Target for predicted rows
pred = pred.copy()
pred["Target"] = pred["Predicted_Target"]

# Drop Predicted_Target so columns match exactly
pred = pred.drop(columns=["Predicted_Target"])

# Combine original filled rows + predicted rows
combined = pd.concat([orig_filled, pred], ignore_index=True)

# Sort by Week for readability
combined = combined.sort_values("Week")

combined

# Coerce mixed-type columns to string for consistent Arrow conversion
# (orig table has comma-formatted strings, pred table has numerics)
for col in combined.columns:
    if combined[col].dtype == 'object':
        combined[col] = combined[col].astype(str)

combined_spark = spark.createDataFrame(combined)

combined_spark.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(
    "workspace.default.nfl_survivor_full_targets"
)

In [0]:
# Load merged table (original + predicted)
df = spark.table("workspace.default.nfl_survivor_full_targets").toPandas()

# Clean column names
df.columns = (
    df.columns
    .str.replace(' ', '_', regex=False)
    .str.replace('[,;{}()\n\t=]', '_', regex=True)
)

# Create a unified risk column
# If Predicted_Target exists, use it
# Otherwise use Target
df["RiskScore"] = df["Predicted_Target"] if "Predicted_Target" in df.columns else df["Target"]

# If both exist (merged table), prefer Predicted_Target when available
if "Predicted_Target" in df.columns:
    df["RiskScore"] = df["Predicted_Target"].fillna(df["Target"])


In [0]:
current_week_raw = dbutils.widgets.get("current_week").strip().lower()
if current_week_raw in ("", "auto"):
    # Set by the Ingest Odds task in the job; when run by hand, ask ESPN directly
    try:
        current_week = int(dbutils.jobs.taskValues.get(taskKey="ingest_odds", key="current_week"))
    except Exception:
        current_week = None
    if current_week is None:
        try:
            # Written by fetch_nfl_data.py when the workspace has no internet
            with open("/Volumes/workspace/default/nfl_survivor/current_week.txt") as f:
                current_week = int(f.read().strip())
        except Exception:
            current_week = None
    if current_week is None:
        import requests
        board = requests.get("https://site.api.espn.com/apis/site/v2/sports/football/nfl/scoreboard", timeout=20).json()
        current_week = int(board.get("week", {}).get("number", 1))
        if board.get("season", {}).get("type") == 1:
            current_week = 1
else:
    current_week = int(current_week_raw)
print(f"Using current_week = {current_week}")
buybacks_used = int(dbutils.widgets.get("buybacks_used"))

df = combined.copy()
# Make sure odds sort as numbers, not text
df["Target"] = pd.to_numeric(df["Target"], errors="coerce")
df["Week"] = pd.to_numeric(df["Week"], errors="coerce").astype(int)


In [0]:
df.columns = (
    df.columns
    .str.replace(' ', '_', regex=False)
    .str.replace('[,;{}()\n\t=]', '_', regex=True)
)

In [ ]:
# Settings for the odds-change simulation
dbutils.widgets.text("odds_change_pct", "15", "Typical odds change, % pts (game 4 weeks out)")
dbutils.widgets.text("simulations", "2000", "Number of simulations")
ODDS_CHANGE_PCT = float(dbutils.widgets.get("odds_change_pct"))
N_SIMS = int(dbutils.widgets.get("simulations"))


In [ ]:
# Win probabilities for every remaining week
import numpy as np
import pandas as pd
from scipy.optimize import linear_sum_assignment

LAST_WEEK = 16  # pool ends after this week


def moneyline_to_prob(ml):
    ml = float(ml)
    return (-ml) / ((-ml) + 100) if ml < 0 else 100 / (ml + 100)


games = df[(df["Week"] >= current_week) & (df["Week"] <= LAST_WEEK)].copy()
games["Target"] = pd.to_numeric(games["Target"], errors="coerce")
games = games.dropna(subset=["Target"])
games["RawProb"] = games["Target"].apply(moneyline_to_prob)
games["Opp"] = games["Team_Opp"].astype(str).str.lstrip("@")

# Remove the bookmaker's margin: scale each side so the two teams in a game add up to 100%
opp_prob = games[["Week", "Team", "RawProb"]].rename(columns={"Team": "Opp", "RawProb": "OppRawProb"})
games = games.merge(opp_prob, on=["Week", "Opp"], how="left")
games["WinProb"] = np.where(games["OppRawProb"].notna(),
                            games["RawProb"] / (games["RawProb"] + games["OppRawProb"]),
                            games["RawProb"])

# Real sportsbook line, or a line the model predicted? Predicted ones are less certain.
real_lines = set(zip(orig_filled["Week"].astype(int), orig_filled["Team"]))
games["Predicted"] = [(int(w), t) not in real_lines for w, t in zip(games["Week"], games["Team"])]

games = games[~games["Team"].isin(used_teams_manual)]
games = games.sort_values("WinProb", ascending=False).drop_duplicates(["Week", "Team"])

weeks = list(range(current_week, LAST_WEEK + 1))
teams = sorted(games["Team"].unique())
P = np.zeros((len(weeks), len(teams)))          # today's win probability (0 = bye)
PRED = np.ones((len(weeks), len(teams)), bool)  # True where the line is a model prediction
for r in games.itertuples():
    i, j = weeks.index(int(r.Week)), teams.index(r.Team)
    P[i, j], PRED[i, j] = r.WinProb, r.Predicted

buybacks_remaining = max(0, 2 - buybacks_used)
opp_lookup = games.set_index(["Week", "Team"])["Team_Opp"].to_dict()
ml_lookup = games.set_index(["Week", "Team"])["Target"].to_dict()
print(f"Weeks {current_week}-{LAST_WEEK}: {len(weeks)} picks left, {len(teams)} teams available, "
      f"{buybacks_remaining} buyback(s) left. Already used: {', '.join(used_teams_manual) or 'none'}")


In [ ]:
# Optimizer helpers
def best_plan(Pm, banned_col=None):
    """Best team for each week (each team once) that maximizes P(win every week).
    Multiplying probabilities = adding logs, so the Hungarian algorithm solves it exactly."""
    if Pm.shape[0] == 0:
        return np.array([], int), np.array([])
    cost = -np.log(np.clip(Pm, 1e-9, 1))
    cost[Pm <= 0] = 1e6
    if banned_col is not None:
        cost[:, banned_col] = 1e6
    rows, cols = linear_sum_assignment(cost)
    cols = cols[np.argsort(rows)]
    return cols, Pm[np.arange(Pm.shape[0]), cols]


def survival(probs, losses_allowed=0):
    """P(lose no more than `losses_allowed` games); each buyback covers one loss."""
    dist = np.zeros(len(probs) + 2)
    dist[0] = 1.0
    for p in probs:
        dist[1:] = dist[1:] * p + dist[:-1] * (1 - p)
        dist[0] *= p
    return float(dist[: losses_allowed + 1].sum())


In [ ]:
# Simulate future odds changes and re-plan in each one
# Each simulation nudges every future game's odds (on the log-odds scale, so 95% can't go past 100%).
# The nudge grows with how far away the game is, and predicted lines move twice as much as real ones.
# ODDS_CHANGE_PCT = typical (1 standard deviation) change in win probability, in percentage points,
# for a predicted line 4 weeks out. Each week you re-plan with the new odds, so in every simulation
# the rest of the season is re-solved; a pick this week is good if it leaves you strong options
# in most of the possible futures, not just under today's odds.
rng = np.random.default_rng(16)
weeks_out = np.arange(len(weeks))[:, None]
sigma = (ODDS_CHANGE_PCT / 100) / 0.1875 * np.sqrt(weeks_out / 4) * np.where(PRED, 1.0, 0.5)
sigma[0] = 0  # this week's odds are what they are
logit = np.log(np.clip(P, 1e-6, 1 - 1e-6) / (1 - np.clip(P, 1e-6, 1 - 1e-6)))

candidates = [j for j in range(len(teams)) if P[0, j] > 0]
surv0 = np.zeros((N_SIMS, len(candidates)))
survk = np.zeros((N_SIMS, len(candidates)))
future_counts = {j: {} for j in candidates}  # candidate -> {(week, team): count}

for s in range(N_SIMS):
    Ps = 1 / (1 + np.exp(-(logit + rng.normal(0, 1, P.shape) * sigma)))
    Ps[P <= 0] = 0
    for k, j in enumerate(candidates):
        cols, probs = best_plan(Ps[1:], banned_col=j)
        plan_probs = np.concatenate([[P[0, j]], probs])
        surv0[s, k] = np.prod(plan_probs)
        survk[s, k] = survival(plan_probs, buybacks_remaining)
        fc = future_counts[j]
        for wi, c in enumerate(cols, start=1):
            fc[(weeks[wi], teams[c])] = fc.get((weeks[wi], teams[c]), 0) + 1

score = survk if buybacks_remaining > 0 else surv0
best_each_sim = score.argmax(axis=1)

# Today's-odds plan for comparison (no simulation)
today = []
for j in candidates:
    cols, probs = best_plan(P[1:], banned_col=j)
    today.append(np.prod(np.concatenate([[P[0, j]], probs])))

options_df = pd.DataFrame({
    "Team": [teams[j] for j in candidates],
    "Team_Opp": [opp_lookup.get((current_week, teams[j])) for j in candidates],
    "WinProb_ThisWeek": [P[0, j] for j in candidates],
    "Expected_Survival": surv0.mean(axis=0),
    "Expected_Survival_With_Buybacks": survk.mean(axis=0),
    "Best_In_Pct_Of_Sims": np.bincount(best_each_sim, minlength=len(candidates)) / N_SIMS,
    "Survival_At_Todays_Odds": today,
}).sort_values(["Expected_Survival_With_Buybacks" if buybacks_remaining else "Expected_Survival",
                "Best_In_Pct_Of_Sims"], ascending=False).reset_index(drop=True)

best_team = options_df.loc[0, "Team"]
best_j = teams.index(best_team)


In [ ]:
# Who to pick this week
this_week_df = options_df.head(10)
print(f"PICK FOR WEEK {current_week}: {best_team} ({opp_lookup.get((current_week, best_team))})")
print(f"Expected chance of surviving through week {LAST_WEEK}: {options_df.loc[0, 'Expected_Survival']:.1%}"
      + (f" (with {buybacks_remaining} buyback(s): {options_df.loc[0, 'Expected_Survival_With_Buybacks']:.1%})"
         if buybacks_remaining else ""))
print(f"Best choice in {options_df.loc[0, 'Best_In_Pct_Of_Sims']:.0%} of {N_SIMS} simulated futures")
this_week_df


In [ ]:
# Likely future picks
rows = [{"Week": current_week, "Team": best_team, "Team_Opp": opp_lookup.get((current_week, best_team)),
         "Score": ml_lookup.get((current_week, best_team)), "WinProb": P[0, best_j],
         "Pct_Of_Sims": 1.0, "Type": "Pick now"}]
fc = future_counts[best_j]
for wk in weeks[1:]:
    top = sorted(((n, t) for (w, t), n in fc.items() if w == wk), reverse=True)[:3]
    for rank, (n, t) in enumerate(top):
        rows.append({"Week": wk, "Team": t, "Team_Opp": opp_lookup.get((wk, t)),
                     "Score": ml_lookup.get((wk, t)), "WinProb": P[weeks.index(wk), teams.index(t)],
                     "Pct_Of_Sims": n / N_SIMS, "Type": "Most likely" if rank == 0 else "Alternative"})
final_df = pd.DataFrame(rows)
final_df


In [0]:
import pandas as pd

# Optional widget for interactive team selection
dbutils.widgets.text("team_lookup", "", "Team to Inspect")
team_lookup = dbutils.widgets.get("team_lookup").strip()

# If you prefer hard-coding, uncomment:
# team_lookup = "DEN"

if team_lookup == "":
    print("Please enter a team abbreviation in the widget.")
else:
    # Convert moneyline to probability
    def moneyline_to_prob(ml):
        ml = float(ml)
        if ml < 0:
            return (-ml) / ((-ml) + 100)
        else:
            return 100 / (ml + 100)

    team_df = df[df["Team"] == team_lookup].copy()

    # Clean and compute probability
    team_df["Target"] = team_df["Target"].astype(float)
    team_df["WinProb"] = team_df["Target"].apply(moneyline_to_prob)

    # Sort by week
    team_df = team_df.sort_values("Week")

    # Select useful columns
    display(team_df[[
        "Week",
        "Team",
        "Team_Opp",
        "Home",
        "Target",
        "WinProb"
    ]])


In [0]:
final_spark = spark.createDataFrame(final_df)
final_spark.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("workspace.default.nfl_survivor_optimal_picks")

# Top 10 options for this week, with the survival odds each one leads to
spark.createDataFrame(this_week_df).write.mode("overwrite").option("overwriteSchema", "true") \
    .saveAsTable("workspace.default.nfl_survivor_pick_options")

# Show the results in the job run output
print(f"PICK FOR WEEK {current_week}: {best_team}  |  expected survival through week {LAST_WEEK}: "
      f"{options_df.loc[0, 'Expected_Survival']:.1%} (with buybacks: {options_df.loc[0, 'Expected_Survival_With_Buybacks']:.1%})")
display(this_week_df)
display(final_df)
